In [4]:
import numpy as np
import pandas as pd
from scipy.stats import norm, chi2
from arch import arch_model

np.random.seed(42)

# --------------------
# 1. Simulate data (2 years of daily returns)
# --------------------
T = 500                       # ~2 years of trading days
mu_true = 0.0003
sigma_true = 0.02
rets = mu_true + sigma_true * np.random.randn(T)  # synthetic daily returns
losses = -rets                                    # define loss = -return
alpha = 0.95                                      # VaR confidence
p = 1 - alpha
window = 250                                      # rolling window length
start = window

# --------------------
# 2. Fit GARCH(1,1) for conditional vol
# --------------------
am = arch_model(rets * 100, vol='Garch', p=1, q=1, dist='normal')
res = am.fit(disp="off")
garch_sigma = res.conditional_volatility / 100.0  # back to return units

# --------------------
# 3. VaR models (1-day horizon)
# --------------------
def rolling_historical_var(losses, alpha, window):
    VaR = np.full_like(losses, np.nan)
    for t in range(window, len(losses)):
        VaR[t] = np.quantile(losses[t-window:t], alpha)
    return VaR

def vc_var_const(losses, alpha, window):
    z = norm.ppf(alpha)
    sigma_const = losses[:window].std()
    VaR = np.full_like(losses, np.nan)
    VaR[window:] = z * sigma_const
    return VaR

def vc_var_garch(garch_sigma, alpha, window):
    z = norm.ppf(alpha)
    VaR = np.full_like(garch_sigma, np.nan)
    for t in range(window, len(garch_sigma)):
        VaR[t] = z * garch_sigma[t-1]  # use prev day's conditional vol
    return VaR

def mc_var_const(mu, sigma, alpha, window, T, n_sim=10000):
    z = norm.ppf(alpha)
    VaR = np.full(T, np.nan)
    # for constant vol, MC VaR is same each day (Gaussian)
    VaR[window:] = z * sigma
    return VaR

def mc_var_garch(garch_sigma, alpha, window, n_sim=10000):
    VaR = np.full_like(garch_sigma, np.nan)
    for t in range(window, len(garch_sigma)):
        sigma_t = garch_sigma[t-1]
        sims = -(mu_true + sigma_t * np.random.randn(n_sim))
        VaR[t] = np.quantile(sims, alpha)
    return VaR

VaR_hist          = rolling_historical_var(losses, alpha, window)
VaR_vc_const      = vc_var_const(losses, alpha, window)
VaR_vc_garch      = vc_var_garch(garch_sigma, alpha, window)
sigma_const_MC    = losses[:window].std()
VaR_mc_const      = mc_var_const(mu_true, sigma_const_MC, alpha, window, T)
VaR_mc_garch      = mc_var_garch(garch_sigma, alpha, window)

# --------------------
# 4. Kupiec & Christoffersen tests
# --------------------
def kupiec_test(exceed, alpha):
    T = len(exceed)
    x = exceed.sum()
    if x == 0 or x == T:
        # edge case: LRuc undefined; treat as extreme miss
        return np.nan, 0.0
    pi_hat = x / T
    pi0 = 1 - alpha
    num = (1 - pi0)**(T - x) * pi0**x
    den = (1 - pi_hat)**(T - x) * pi_hat**x
    LR_uc = -2 * np.log(num / den)
    p_value = 1 - chi2.cdf(LR_uc, df=1)
    return LR_uc, p_value

def christoffersen_test(exceed, alpha):
    T = len(exceed)
    x = exceed.sum()
    # Unconditional part
    LR_uc, p_uc = kupiec_test(exceed, alpha)

    # Transition counts for independence
    I = exceed.astype(int)
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        prev, curr = I[t-1], I[t]
        if prev == 0 and curr == 0: n00 += 1
        elif prev == 0 and curr == 1: n01 += 1
        elif prev == 1 and curr == 0: n10 += 1
        else: n11 += 1

    if (n00 + n01) == 0 or (n10 + n11) == 0:
        return LR_uc, p_uc, np.nan, 0.0, np.nan, 0.0

    pi01 = n01 / (n00 + n01)
    pi11 = n11 / (n10 + n11)
    pi   = (n01 + n11) / (n00 + n01 + n10 + n11)

    L0 = ((1 - pi)**(n00 + n10)) * (pi**(n01 + n11))
    L1 = ((1 - pi01)**n00) * (pi01**n01) * ((1 - pi11)**n10) * (pi11**n11)
    LR_ind = -2 * np.log(L0 / L1)
    LR_cc  = LR_uc + LR_ind
    p_ind  = 1 - chi2.cdf(LR_ind, df=1)
    p_cc   = 1 - chi2.cdf(LR_cc, df=2)
    return LR_uc, p_uc, LR_ind, p_ind, LR_cc, p_cc

# --------------------
# 5. Backtesting and comparison
# --------------------
methods = {
    "Hist"        : VaR_hist,
    "VC_const"    : VaR_vc_const,
    "VC_GARCH"    : VaR_vc_garch,
    "MC_const"    : VaR_mc_const,
    "MC_GARCH"    : VaR_mc_garch,
}

results = []

for name, VaR in methods.items():
    mask = ~np.isnan(VaR)
    L = losses[mask]
    V = VaR[mask]
    exceed = (L > V).astype(int)   # 1 if loss exceeds VaR
    exc_rate = exceed.mean()

    LR_uc, p_uc, LR_ind, p_ind, LR_cc, p_cc = christoffersen_test(exceed, alpha)

    results.append({
        "Model"        : name,
        "Obs"          : len(L),
        "Exc_Rate"     : exc_rate,
        "p_uc"         : p_uc,
        "p_ind"        : p_ind,
        "p_cc"         : p_cc,
    })

df_res = pd.DataFrame(results)
print(df_res)


      Model  Obs  Exc_Rate      p_uc     p_ind      p_cc
0      Hist  250     0.056  0.669066  0.196383  0.396181
1  VC_const  250     0.040  0.452912  0.360238  0.496482
2  VC_GARCH  250     0.036  0.286022  0.411259  0.403852
3  MC_const  250     0.040  0.452912  0.360238  0.496482
4  MC_GARCH  250     0.036  0.286022  0.411259  0.403852
